In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Euclidean Division

The function `divMod` takes two arguments:
- `m` is a natural number,
- `n` is a natural number such that $n > 0$.

It returns the unique pair $(q, r)$
that satisfies the following two formulas:
1. $m = q \cdot n + r$,
2. $r < n$.

Hence $q$ is the *quotient* and $r$ is the *remainder* of the division of $m$ by $n$.  The function uses only division by $2$, which is a cheap operation for binary numbers.

**Implementation:** If $m < n$, then $q = 0$ and $r = m$.  Otherwise, the quotient $q'$ and remainder $r'$ of $m / 2$ (rounded down) divided by $n$ are computed recursively.  We have $m = 2 \cdot (m / 2) + m \,\texttt{%}\, 2$ and therefore 
$$ m = 2 \cdot (q' \cdot n + r') + m \,\texttt{%}\, 2 = 2 \cdot q' \cdot n + 2 \cdot r' + m \,\texttt{%}\, 2. $$
- If $2 \cdot r' + m \,\texttt{%}\, 2 < n$, then $q = 2 \cdot q'$ and $r = 2 \cdot r' + m \,\texttt{%}\, 2$.
- Otherwise, we have $n \leq 2 \cdot r' + m \,\texttt{%}\, 2 < 2 \cdot n$ and therefore $q = 2 \cdot q' + 1$ and $r = 2 \cdot r' + m \,\texttt{%}\, 2 - n$.

As we want to test the function with numbers up to $10^{12}$, the arguments have the type `int64`, i.e. they are 64-bit integers.  Literals of type `int64` are written with the suffix `L`, e.g. `0L`.  In F#, the operator `/` applied to integers computes the integer division, which corresponds to the operator `//` in Python.

In [2]:
let rec divMod (m: int64) (n: int64) : int64 * int64 =
    if m < n then
        (0L, m)
    else
        let (q, r) = divMod (m / 2L) n
        if 2L * r + m % 2L < n then
            (2L * q     , 2L * r + m % 2L)
        else
            (2L * q + 1L, 2L * r + m % 2L - n)

In [3]:
divMod 100L 7L

(14L, 2L)

We test this function using random numbers.  `System.Random(42)` creates a random number generator with the *seed* `42`.  Using a fixed seed ensures that the tests are reproducible.  The method `rnd.NextInt64(a, b)` returns a random number $x$ such that $a \leq x < b$.

In [4]:
let rnd = System.Random(42)

In [5]:
for _ in 1 .. 1_000_000 do
    let m = rnd.NextInt64(0L, 1_000_000_000_000L)
    let n = rnd.NextInt64(1L, 1_000_000_000L)
    let (q, r) = divMod m n
    if not (m = q * n + r && r < n) then
        failwithf "m = %d, n = %d, q = %d, r = %d" m n q r
printfn "All tests have been successful!"

All tests have been successful!